# PA1: Notebook de Inferência de Instâncias

**Entregável Oficial do PA1:**
> *"Recebe o caminho de uma imagem qualquer, devolve a máscara de instâncias colorida e a contagem. Roda sem retreinar."*

Este notebook carrega os pesos treinados do modelo final (`checkpoints/best_model.pth`) e processa qualquer imagem de entrada fornecida.


In [1]:
import os
import sys
sys.path.append("..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import label as ndimage_label

from src.utils import colorize_instances, overlay_mask_on_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de inferência configurado: {device}")

Dispositivo de inferência configurado: cuda


## 1. Carregamento do Modelo Treinado


In [2]:
def load_trained_model(checkpoint_path: str = "../checkpoints/best_model.pth"):
    """Carrega a arquitetura e os pesos do modelo treinado para inferência."""
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Checkpoint não encontrado em {checkpoint_path}. "
            "Certifique-se de salvar os pesos do modelo final antes de rodar a inferência."
        )
    
    from src.models import build_model
    model = build_model()
    state_dict = torch.load(checkpoint_path, map_location=device)
    if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
        state_dict = state_dict["model_state_dict"]
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model

# Tentar carregar modelo se checkpoint existir
checkpoint_file = "../checkpoints/best_model.pth"
if os.path.exists(checkpoint_file):
    model = load_trained_model(checkpoint_file)
    print("Modelo final carregado com sucesso!")
else:
    model = None
    print(f"[Aviso] Checkpoint {checkpoint_file} ainda não encontrado. O notebook executará assim que os pesos forem gerados.")

[Aviso] Checkpoint ../checkpoints/best_model.pth ainda não encontrado. O notebook executará assim que os pesos forem gerados.


## 2. Função de Inferência em Imagem Arbitrária


In [3]:
def predict_instances(
    image_path: str,
    model: torch.nn.Module,
    threshold: float = 0.50,
    target_size: tuple = (256, 256),
):
    """Recebe o caminho de uma imagem qualquer e retorna a máscara de instâncias colorida e a contagem."""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Imagem não encontrada: {image_path}")
    
    # 1. Carregar imagem original
    with Image.open(image_path) as pil_img:
        img_rgb = pil_img.convert("RGB")
        orig_w, orig_h = img_rgb.size
        resized_img = img_rgb.resize((target_size[1], target_size[0]), Image.Resampling.BILINEAR)
        img_np = np.array(resized_img, dtype=np.float32) / 255.0
    
    # 2. Tensor PyTorch (1, 3, H, W)
    tensor_in = torch.from_numpy(img_np.transpose(2, 0, 1)).unsqueeze(0).float().to(device)
    
    # 3. Predição da rede
    with torch.no_grad():
        logits = model(tensor_in)
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    
    # 4. Extração de instâncias (componentes conexos / watershed)
    binary = (prob >= threshold).astype(np.uint8)
    instance_mask, num_instances = ndimage_label(binary)
    
    # 5. Redimensionar máscara de volta para o tamanho original
    pil_inst = Image.fromarray(instance_mask.astype(np.int32))
    pil_inst_orig = pil_inst.resize((orig_w, orig_h), Image.Resampling.NEAREST)
    final_instance_mask = np.array(pil_inst_orig, dtype=np.int64)
    
    orig_img_np = np.array(img_rgb, dtype=np.float32) / 255.0
    colored_mask = colorize_instances(final_instance_mask, seed=42)
    overlay = overlay_mask_on_image(orig_img_np, colored_mask, alpha=0.50)
    
    return {
        "original_image": orig_img_np,
        "instance_mask": final_instance_mask,
        "colored_mask": colored_mask,
        "overlay": overlay,
        "count": int(num_instances),
    }

## 3. Demonstração e Visualização de Resultados

Defina abaixo o caminho de qualquer imagem para obter a segmentação e contagem:


In [4]:
# Exemplo com uma imagem da base real ou imagem externa qualquer
sample_image_path = "../data/raw/stage1_train/00071198d059ba7f5914a526d124d28e6d010c92466da21d4a04cd5413362552/images/00071198d059ba7f5914a526d124d28e6d010c92466da21d4a04cd5413362552.png"

if model is not None and os.path.exists(sample_image_path):
    result = predict_instances(sample_image_path, model=model)
    
    print("=" * 60)
    print(f"CONTAGEM TOTAL DE OBJETOS DETECTADOS: {result['count']} núcleos")
    print("=" * 60)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(result["original_image"])
    axes[0].set_title("Imagem de Entrada", fontsize=12)
    axes[0].axis("off")
    
    axes[1].imshow(result["colored_mask"])
    axes[1].set_title(f"Máscara de Instâncias Colorida\n({result['count']} núcleos)", fontsize=12)
    axes[1].axis("off")
    
    axes[2].imshow(result["overlay"])
    axes[2].set_title("Sobreposição (Overlay)", fontsize=12)
    axes[2].axis("off")
    
    plt.suptitle(f"Resultado da Inferência — Contagem: {result['count']}", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Aguardando treinamento do modelo para executar a demonstração visual.")

Aguardando treinamento do modelo para executar a demonstração visual.
